In [65]:
!pip install pennylane torch transformers scikit-learn numpy

In [66]:
import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report

N_QUBITS = 5
N_LAYERS = 6 # circuit depth=4 as per paper
dev = qml.device("default.qubit", wires=N_QUBITS)

# New section

In [67]:
@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    # Step 1: Hadamard gates — creates superposition (paper Figure 1)
    for i in range(N_QUBITS):
        qml.Hadamard(wires=i)

    # Step 2: Angle encoding with RY gates (paper Equation 2)
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")

    # Step 3: Variational layers — RY + CNOT entanglement (paper Figure 1)
    for layer in range(N_LAYERS):
        # Trainable RY rotations
        for i in range(N_QUBITS):
            qml.RY(weights[layer][i], wires=i)
        # CNOT entanglement chain (paper: "fully entangled state")
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[N_QUBITS - 1, 0])  # close the loop

    # Step 4: Measure all qubits — PauliZ expectation (paper Equation 4)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

In [68]:
# Wrap quantum circuit as torch layer
weight_shapes = {"weights": (N_LAYERS, N_QUBITS)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

class HybridQuantumBERT(nn.Module):
    def __init__(self, input_dim=768, n_qubits=N_QUBITS, n_classes=2):
        super().__init__()

        # Classical pre-processing: 768 → n_qubits (paper: L_768→nq)
        self.pre_classical = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_qubits),
            nn.Tanh()  # squash to [-1,1] before angle encoding
        )

        # Quantum layer
        self.quantum = qlayer

        # Classical post-processing: n_qubits → 2 (paper: L_nq→2)
        self.post_classical = nn.Sequential(
            nn.Linear(n_qubits, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, n_classes)
        )

    def forward(self, x):
        # Classical pre-layer
        x = self.pre_classical(x)          # (batch, n_qubits)

        # Quantum layer — process each sample
        x = self.quantum(x)                # (batch, n_qubits)

        # Classical post-layer
        x = self.post_classical(x)         # (batch, 2)
        return x

model = HybridQuantumBERT(input_dim=768, n_qubits=N_QUBITS, n_classes=2)

In [69]:
import numpy as npy  # plain numpy for loading
from sklearn.utils.class_weight import compute_class_weight

# Load original BERT embeddings — no PCA needed now
X_train = npy.load("X_train_mbert.npy")   # (5798, 768)
X_test  = npy.load("X_test_mbert.npy")    # (1450, 768)
y_train = npy.load("y_train_binary.npy")
y_test  = npy.load("y_test_binary.npy")

# Scale
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# Class weights for imbalance
cw = compute_class_weight('balanced', classes=npy.unique(y_train), y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float32)

In [70]:
from torch.utils.data import TensorDataset, DataLoader

# Paper settings: lr=1e-3, batch=32, epochs=10
# quantum_weight=0.01 applied via separate param groups
optimizer = torch.optim.Adam([
    {'params': model.pre_classical.parameters(),  'lr': 1e-4},
    {'params': model.quantum.parameters(),         'lr': 0.005},  # quantum weight
    {'params': model.post_classical.parameters(), 'lr': 1e-4}
])

criterion = nn.CrossEntropyLoss(weight=class_weights)

dataset = TensorDataset(X_train_t, y_train_t)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)

epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        out  = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f}")

Epoch  1/30 | Loss: 0.6889
Epoch  2/30 | Loss: 0.6770
Epoch  3/30 | Loss: 0.6612
Epoch  4/30 | Loss: 0.6464
Epoch  5/30 | Loss: 0.6205
Epoch  6/30 | Loss: 0.6014
Epoch  7/30 | Loss: 0.5816
Epoch  8/30 | Loss: 0.5579
Epoch  9/30 | Loss: 0.5396
Epoch 10/30 | Loss: 0.5240
Epoch 11/30 | Loss: 0.5020
Epoch 12/30 | Loss: 0.4920
Epoch 13/30 | Loss: 0.4739
Epoch 14/30 | Loss: 0.4517
Epoch 15/30 | Loss: 0.4299
Epoch 16/30 | Loss: 0.4163
Epoch 17/30 | Loss: 0.3953
Epoch 18/30 | Loss: 0.3760
Epoch 19/30 | Loss: 0.3558
Epoch 20/30 | Loss: 0.3356
Epoch 21/30 | Loss: 0.3181
Epoch 22/30 | Loss: 0.2980
Epoch 23/30 | Loss: 0.2806
Epoch 24/30 | Loss: 0.2594
Epoch 25/30 | Loss: 0.2536
Epoch 26/30 | Loss: 0.2311
Epoch 27/30 | Loss: 0.2197
Epoch 28/30 | Loss: 0.2009
Epoch 29/30 | Loss: 0.1911
Epoch 30/30 | Loss: 0.1810


In [71]:
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    preds  = torch.argmax(logits, dim=1).numpy()

print(f"Test accuracy: {accuracy_score(y_test, preds):.4f}")
print(classification_report(y_test, preds))

Test accuracy: 0.6903
              precision    recall  f1-score   support

           0       0.78      0.70      0.74       896
           1       0.58      0.68      0.63       554

    accuracy                           0.69      1450
   macro avg       0.68      0.69      0.68      1450
weighted avg       0.70      0.69      0.69      1450

